# Factorio YOLO v0 Notebook

This notebook is for model inference in Hub/Colab/local environments.

## Environment
- Python 3.10+
- Dependencies are installed in the next cell
- Model is loaded from Hugging Face Hub using a fixed `revision`

Release note: update `revision` when a new model tag is published.


In [ ]:
%pip install -q ultralytics==8.3.0 huggingface_hub==0.34.4 onnxruntime==1.22.0 matplotlib==3.9.2 pillow==10.4.0


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

model_repo_id = "proj-airi/factorio-yolo-v0"
model_revision = "v0.1.0"  # Update on each release

dataset_repo_id = "proj-airi/factorio-yolo-dataset-v0"
dataset_revision = model_revision  # Keep aligned with model version
dataset_example_path = "examples/demo.jpg"  # Update to a real file in dataset repo

pt_path = hf_hub_download(repo_id=model_repo_id, filename="best.pt", revision=model_revision)

print(f"PyTorch weights: {pt_path}")

model = YOLO(pt_path)


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

image_path = Path(hf_hub_download(
    repo_id=dataset_repo_id,
    repo_type="dataset",
    filename=dataset_example_path,
    revision=dataset_revision,
))

# Or you can use your own image path instead
# image_path = Path("path/to/your-square-image.jpg")

if not image_path.exists():
    print("Set a valid image_path, then rerun this cell.")
else:
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        w, h = image.size
        if w != h:
            raise ValueError(f"Image must be square (W == H), got {w}x{h}.")

        results = model.predict(source=image, save=False)
    result = results[0]

    annotated_bgr = result.plot()
    annotated_rgb = annotated_bgr[..., ::-1]

    plt.figure(figsize=(8, 8))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title("Predictions")
    plt.show()

    for box in result.boxes:
        cls_id = int(box.cls[0].item())
        conf = float(box.conf[0].item())
        print(result.names[cls_id], f"{conf:.2f}")
